# Imports

In [1]:
import os
import sys

import numpy as np
import torch
from torchvision.transforms import GaussianBlur
from IPython.core.pylabtools import figsize
from mpmath.identification import transforms
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import RandomApply

from utils.checkpoint import load_checkpoint, save_checkpoint
from utils.data import get_dataloaders, get_img_from_loader
from evaluate import evaluate, evaluate_with_uncertainty

sys.path.append("..")

from models.lenet import Net as LeNet
from config import Config

from utils.corruptions import gaussian_blur, test_on_corruptions, corruptions_uncertainty
from utils.data import get_img_from_loader



In [2]:
config = Config()
device = config.device

# Prepare data
_, val_loader, test_loader = get_dataloaders(
    data_dir="../data",
    batch_size=config.test_batch_size,
    num_workers=config.num_workers,
    use_cuda=torch.cuda.is_available(),
)

# Model
lenet_model = LeNet(
    prior_sigma1=config.prior_sigma1,
    prior_sigma2=config.prior_sigma2,
    prior_pi=config.prior_pi,
    num_classes=config.num_classes,
).to(device)

# Optimizer (needed to load checkpoint)
optimizer = torch.optim.Adam(lenet_model.parameters(), lr=config.learning_rate)
# Load weights
# config.model_name = 'mnist_bayesian_lenet'
config.model_name = 'lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1'
epoch = load_checkpoint(lenet_model, optimizer, f'{config.checkpoint_path}/{config.model_name}/{config.get_checkpoint_name(190, date="20260129")}', device)
# load_checkpoint(lenet_model, optimizer, f'{config.checkpoint_path}/{config.get_checkpoint_name(475, date="20251201")}', device)


[checkpoint] Loaded from ../checkpoints/lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1/lenet_mnist_lrp1em04_logprior10_logprior2m6_priorpip5_v1_epoch_190_20260129.pth, starting at epoch 191


# Зависимость неопределенности модели от правильности предсказаний

In [5]:
# Анализ корреляции между неопределенностью и правильностью предсказаний для разных уровней размытия
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.transforms import GaussianBlur
from utils.uncertainty import mc_predict, quantify_uncertainties

T = 10
kernel_sizes = [0, 1, 3, 5, 7, 9]
# kernel_sizes = [0, 1, 3, 7]
correlations = []
mean_unc_correct = []
mean_unc_incorrect = []
mean_aunc_correct = []
mean_aunc_incorrect = []
mean_eunc_correct = []
mean_eunc_incorrect = []

lenet_model.eval()

for ks in kernel_sizes:
    if ks == 0:
        current_loader = test_loader
        desc = "Original"
    else:
        current_loader = get_dataloaders(
            data_dir="../data",
            batch_size=config.test_batch_size,
            num_workers=config.num_workers,
            use_cuda=torch.cuda.is_available(),
            extra_transforms=[GaussianBlur(ks)]
        )[2]
        desc = f"Blur {ks}"

    all_correct = []
    all_conf = []
    all_uncertainties = []
    all_aleatoric = []
    all_epistemic = []

    print(f"Processing {desc}...")
    for x, y in tqdm(current_loader, leave=False):
        x, y = x.to(device), y.to(device)
        
        # Оценка неопределенности
        mc_preds = mc_predict(lenet_model, x, mc_samples=T)
        preds, (total_unc_mat, alea_unc_mat, _) = quantify_uncertainties(mc_preds)
        
        # Извлекаем диагональ (дисперсию для каждого класса) и суммируем
        unc = total_unc_mat.diagonal(dim1=1, dim2=2).sum(-1)
        aunc = alea_unc_mat.diagonal(dim1=1, dim2=2).sum(-1)
        correct = (preds == y).float()
        
        # Softmax probability of the predicted class (mean over MC samples)
        mean_probs = mc_preds.mean(0)  # [batch, num_classes]
        pred_conf = mean_probs[torch.arange(len(preds)), preds]  # [batch]
        true_conf = mean_probs[torch.arange(len(y)), y]  # [batch]

        all_correct.append(correct.cpu().numpy())
        all_conf.append(true_conf.cpu().numpy())
        all_uncertainties.append(unc.cpu().numpy())
        all_aleatoric.append(aunc.cpu().numpy())
        all_epistemic.append((unc - aunc).cpu().numpy())

    c_vec = np.concatenate(all_correct)
    conf_vec = np.concatenate(all_conf)
    u_vec = np.concatenate(all_uncertainties)
    au_vec = np.concatenate(all_aleatoric)
    eu_vec = np.concatenate(all_epistemic)
    eunc_mean = np.mean(eu_vec)
    aunc_mean = np.mean(au_vec)
    print(f"Kernel {ks}: E[epistemic]={eunc_mean:.6f}, E[aleatoric]={aunc_mean:.6f}, ratio={eunc_mean/aunc_mean:.4f}")

    # Считаем корреляцию (softmax confidence of predicted class vs uncertainty)
    corr = np.corrcoef(conf_vec, u_vec)[0, 1]
    correlations.append(corr)
    
    # Считаем среднюю неопределенность для верных и неверных ответов
    mean_unc_correct.append(u_vec[c_vec == 1].mean() if any(c_vec == 1) else 0)
    mean_unc_incorrect.append(u_vec[c_vec == 0].mean() if any(c_vec == 0) else 0)
    mean_aunc_correct.append(au_vec[c_vec == 1].mean() if any(c_vec == 1) else 0)
    mean_aunc_incorrect.append(au_vec[c_vec == 0].mean() if any(c_vec == 0) else 0)
    mean_eunc_correct.append(eu_vec[c_vec == 1].mean() if any(c_vec == 1) else 0)
    mean_eunc_incorrect.append(eu_vec[c_vec == 0].mean() if any(c_vec == 0) else 0)

Processing Original...


Kernel 0: E[epistemic]=0.001152, E[aleatoric]=0.014564, ratio=0.0791
Processing Blur 1...


Kernel 1: E[epistemic]=0.001142, E[aleatoric]=0.014681, ratio=0.0778
Processing Blur 3...


Kernel 3: E[epistemic]=0.001560, E[aleatoric]=0.018663, ratio=0.0836
Processing Blur 5...


Kernel 5: E[epistemic]=0.004292, E[aleatoric]=0.037112, ratio=0.1157
Processing Blur 7...


Kernel 7: E[epistemic]=0.005985, E[aleatoric]=0.050877, ratio=0.1176
Processing Blur 9...


Kernel 9: E[epistemic]=0.006617, E[aleatoric]=0.055240, ratio=0.1198


In [6]:
# Визуализация результатов
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
x_labels = [str(k) for k in kernel_sizes]
x = np.arange(len(kernel_sizes))
titles = ['Total', 'Aleatoric', 'Epistemic']
incorrect = [mean_unc_incorrect, mean_aunc_incorrect, mean_eunc_incorrect]
correct = [mean_unc_correct, mean_aunc_correct, mean_eunc_correct]

# График 1: Коэффициент корреляции
axes[0, 0].plot(kernel_sizes, correlations, marker='o', color='red', linestyle='--')
axes[0, 0].set_title("Correlation (Accuracy vs Uncertainty)")
axes[0, 0].set_xticks(kernel_sizes)
axes[0, 0].set_xlabel("Kernel Size")
axes[0, 0].set_ylabel("Pearson Correlation")
axes[0, 0].grid(True, alpha=0.3)

# График 2: Сравнение неопределенности для Correct vs Incorrect
for i, ax in enumerate(axes.flatten()[1:]):
    width = 0.35
    ax.bar(x - width/2, correct[i], width, label='Correct Predictions', color='green', alpha=0.7)
    ax.bar(x + width/2, incorrect[i], width, label='Incorrect Predictions', color='orange', alpha=0.7)
    ax.set_title(f"Mean {titles[i]} Uncertainty: Correct vs Incorrect")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_xlabel("Kernel Size")
    ax.set_ylabel(f"{titles[i]} Uncertainty")
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

for i, ks in enumerate(kernel_sizes):
    print(f"Kernel {ks}: Correlation = {correlations[i]:.4f}, Unc Ratio (Err/Corr) = {mean_unc_incorrect[i]/mean_unc_correct[i]:.2f}x")
    print(f"\tAlea Unc Ratio (Err/Corr) = {mean_aunc_incorrect[i]/mean_aunc_correct[i]:.2f}x, Unc Ratio (Err/Corr) = {mean_eunc_incorrect[i]/mean_eunc_correct[i]:.2f}x")


Kernel 0: Correlation = -0.7915, Unc Ratio (Err/Corr) = 34.97x
	Alea Unc Ratio (Err/Corr) = 33.57x, Unc Ratio (Err/Corr) = 55.92x
Kernel 1: Correlation = -0.7899, Unc Ratio (Err/Corr) = 33.90x
	Alea Unc Ratio (Err/Corr) = 33.08x, Unc Ratio (Err/Corr) = 45.49x
Kernel 3: Correlation = -0.7743, Unc Ratio (Err/Corr) = 23.72x
	Alea Unc Ratio (Err/Corr) = 23.09x, Unc Ratio (Err/Corr) = 31.89x
Kernel 5: Correlation = -0.7711, Unc Ratio (Err/Corr) = 12.03x
	Alea Unc Ratio (Err/Corr) = 11.64x, Unc Ratio (Err/Corr) = 15.70x
Kernel 7: Correlation = -0.6898, Unc Ratio (Err/Corr) = 8.39x
	Alea Unc Ratio (Err/Corr) = 8.24x, Unc Ratio (Err/Corr) = 9.71x
Kernel 9: Correlation = -0.6669, Unc Ratio (Err/Corr) = 7.42x
	Alea Unc Ratio (Err/Corr) = 7.28x, Unc Ratio (Err/Corr) = 8.60x


In [ ]:
for name, module in lenet_model.named_modules():
    if hasattr(module, 'weight_sigma'):  # или как у тебя называется параметр
        sigma = module.weight_sigma
        # если используется rho-параметризация: sigma = log(1 + exp(rho))
        print(f"{name}: mean(sigma) = {sigma.mean().item():.6f}, max(sigma) = {sigma.max().item():.6f}")